<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/07-joins-deep-dive/01-join-strategies.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 7.1 — Join strategies: BHJ, SMJ, SHJ, BNLJ

We manufacture each physical strategy on synthetic tables and read
the plans. Synthetic because strategy choice is about SIZE — the
`data/` files are all tiny enough to broadcast.

Note: AQE is disabled in this session so `explain()` shows exactly
what will run — with AQE on (the default), the static plan can be
switched at runtime (Module 11).

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("7.1-join-strategies")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "false")   # static plans for teaching
    .config("spark.sql.shuffle.partitions", "8")     # readable plans/UI locally
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# a "big" fact: 2M synthetic order lines, and a small 1k-row dimension
big = spark.range(0, 2_000_000).select(
    col("id").alias("order_id"),
    (col("id") % 1000).alias("cust_key"),
    (F.rand(seed=7) * 100).alias("amount"),
)
dim = spark.range(0, 1000).select(
    col("id").alias("cust_key"),
    F.concat(F.lit("segment_"), (col("id") % 5).cast("string")).alias("segment"),
)
print(big.count(), dim.count())

## 1. BroadcastHashJoin — the fast path

`dim` is far below `spark.sql.autoBroadcastJoinThreshold` (10 MB
default), so Spark broadcasts it automatically. Look for:
`BroadcastHashJoin ... BuildRight` and a `BroadcastExchange` —
and NO `Exchange hashpartitioning` on the big side.

In [ ]:
print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
big.join(dim, "cust_key").explain()

## 2. SortMergeJoin — the scalable default

Kill the threshold and the same join becomes SMJ. Count the cost:
one `Exchange hashpartitioning` per side (both tables shuffled),
then a `Sort` per side, then the merge.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
big.join(dim, "cust_key").explain()

## 3. ShuffledHashJoin — same shuffle, no sort, more risk

With sort-merge preferred by default, SHJ needs a hint. Same two
Exchanges as SMJ, but no `Sort` nodes — instead one side becomes an
in-memory hash table per partition (OOM risk if the estimate lied).

In [ ]:
big.join(dim.hint("shuffle_hash"), "cust_key").explain()

In [ ]:
# the SQL spelling of the same hints (4.1: same plan, different syntax)
big.createOrReplaceTempView("big")
dim.createOrReplaceTempView("dim")
spark.sql("""
    SELECT /*+ SHUFFLE_HASH(dim) */ *
    FROM big JOIN dim USING (cust_key)
""").explain()

## 4. BroadcastNestedLoopJoin — what non-equi conditions cost

A range condition has nothing to hash or sort on. With a small
side, Spark broadcasts it and loops over every (row, row) pair.

In [ ]:
tiers = spark.createDataFrame(
    [(0.0, 25.0, "low"), (25.0, 75.0, "mid"), (75.0, 100.0, "high")],
    "lo DOUBLE, hi DOUBLE, tier STRING",
)
non_equi = big.join(tiers, (col("amount") >= col("lo")) & (col("amount") < col("hi")))
non_equi.explain()

In [ ]:
# It runs — 2M x 3 pair evaluations — but watch the wall clock vs the BHJ.
non_equi.groupBy("tier").count().show()

## Which side CAN be broadcast? Join type decides

The preserved side of an outer join can't be broadcast: a left join
can only broadcast the right side. Ask for the impossible and Spark
silently falls back — the hint is not a command (7.2 digs in).

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

big.join(dim, "cust_key", "left").explain()   # left join: RIGHT side broadcastable — BHJ BuildRight
dim.join(big, "cust_key", "right").explain()  # right join: LEFT side broadcastable — BHJ BuildLeft
dim.join(big, "cust_key", "full").explain()   # full outer: neither side — SMJ, no broadcast possible

## Exercises

1. Predict the strategy before calling `explain()`: (a) `big` join
   `big` on `cust_key`; (b) `dim` join `tiers` on a range condition;
   (c) `big` full-outer `dim`. Verify all three.
2. Force the 2M x 2M self-join to ShuffledHashJoin with a hint. Compare
   wall-clock vs SMJ with `%%time` (or `time.time()`), 3 runs each.
3. Rewrite the `tiers` non-equi join as an equi-join: bucket `amount`
   into an integer key (`floor(amount/25)`) on both sides, join on it,
   and confirm the plan is now a hash join. This "discretize the range"
   trick is a real production pattern.
4. Re-enable AQE, rerun the SMJ cell with the threshold still at -1 —
   then check the Spark UI SQL tab: did the final plan differ from
   `explain()`? (Answer in Module 11: AQE can't broadcast here because
   the threshold forbids it, but it WILL coalesce shuffle partitions.)

In [ ]:
# your exercise code here